In [1]:
from pyscf import gto, scf, dft, lib
import numpy as np

## Setups

In [2]:
mol = gto.Mole(atom="C; H 1 0.94; H 1 0.94 2 104.5", spin=2, charge=2, basis="def2-TZVP").build()

In [3]:
mf = scf.UKS(mol, xc="TPSS0").run()

converged SCF energy = -37.7173135264959  <S^2> = 2.0026514  2S+1 = 3.0017671


In [4]:
arrays = {
    "mo_coeff": np.asarray(mf.mo_coeff, order="C"),
    "mo_occ": np.asarray(mf.mo_occ, order="C"),
    "rdm1": mf.make_rdm1(),
    "coords": mf.grids.coords,
    "weights": mf.grids.weights,
}

In [5]:
mo_coeff = np.asarray(mf.mo_coeff, order="C")
mo_occ = np.asarray(mf.mo_occ, order="C")
rdm1 = mf.make_rdm1()
coords = mf.grids.coords
weights = mf.grids.weights

In [6]:
grids = mf.grids

In [7]:
ni = dft.numint.NumInt()

In [8]:
np.savez("ch2", **arrays)

## Get rho by heterogeneous brakets

In [9]:
# fake a heterogeneous brakets by forming its symmetric dm
dms = mf.mo_coeff[:, :, :4] @ mf.mo_coeff[0, :, :4].T

In [10]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = np.array([ni.eval_rho(mol, ao[0], dms[s], xctype="lda") for s in (0, 1)])
rho_sigma = np.array([ni.eval_rho(mol, ao[0:4], dms[s], xctype="gga") for s in (0, 1)])
rho_tau = np.array([ni.eval_rho(mol, ao[0:4], dms[s], xctype="mgga", with_lapl=False) for s in (0, 1)])
rho_lapl = np.array([ni.eval_rho(mol, ao[0:10], dms[s], xctype="mgga", with_lapl=True) for s in (0, 1)])

In [11]:
rho_rho.shape, lib.fp(rho_rho)

((2, 33736), np.float64(90.19801284927289))

In [12]:
rho_sigma.shape, lib.fp(rho_sigma)

((2, 4, 33736), np.float64(358.29878251752757))

In [13]:
rho_tau.shape, lib.fp(rho_tau)

((2, 5, 33736), np.float64(5531.545589708351))

In [14]:
rho_lapl_ = rho_lapl[:, [0, 1, 2, 3, 5, 4], :]
rho_lapl_.shape, lib.fp(rho_lapl_[:, :5, :]), lib.fp(rho_lapl_[:, 5, :])

((2, 6, 33736), np.float64(5531.545589708351), np.float64(-795547.4132925004))

## Get rho by dm

In [15]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = np.array([ni.eval_rho(mol, ao[0], rdm1[s], xctype="lda") for s in (0, 1)])
rho_sigma = np.array([ni.eval_rho(mol, ao[0:4], rdm1[s], xctype="gga") for s in (0, 1)])
rho_tau = np.array([ni.eval_rho(mol, ao[0:4], rdm1[s], xctype="mgga", with_lapl=False) for s in (0, 1)])
rho_lapl = np.array([ni.eval_rho(mol, ao[0:10], rdm1[s], xctype="mgga", with_lapl=True) for s in (0, 1)])

In [16]:
rho_rho.shape, lib.fp(rho_rho)

((2, 33736), np.float64(90.1600267407406))

In [17]:
rho_sigma.shape, lib.fp(rho_sigma)

((2, 4, 33736), np.float64(369.817842854633))

In [18]:
rho_tau.shape, lib.fp(rho_tau)

((2, 5, 33736), np.float64(5587.859016487361))

In [19]:
rho_lapl_ = rho_lapl[:, [0, 1, 2, 3, 5, 4], :]
rho_lapl_.shape, lib.fp(rho_lapl_[:, :5, :]), lib.fp(rho_lapl_[:, 5, :])

((2, 6, 33736), np.float64(5587.859016487361), np.float64(-783527.5368333756))

## xc_eff

### RHO (LDA)

In [20]:
result = ni.eval_xc_eff("LDA_X", rho_rho, deriv=3, spin=1)

In [21]:
[r.shape for r in result]

[(33736,), (2, 1, 33736), (2, 1, 2, 1, 33736), (2, 1, 2, 1, 2, 1, 33736)]

In [24]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_rho[:, None, :] * weights),
    lib.fp(result[3] * rho_rho[:, None, :] * rho_rho[:, None, None, None, :] * weights),
])
fps

array([-0.00506798,  0.10130377, -0.04175936,  0.02812571])

### SIGMA (GGA)

In [25]:
result = ni.eval_xc_eff("GGA_X_PBE", rho_sigma, deriv=3, spin=1)

In [26]:
[r.shape for r in result]

[(33736,), (2, 4, 33736), (2, 4, 2, 4, 33736), (2, 4, 2, 4, 2, 4, 33736)]

In [28]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_sigma * weights),
    lib.fp(result[3] * rho_sigma * rho_sigma[:, :, None, None, :] * weights),
])
fps

array([ 0.01748262, -0.06882439, -0.09985614,  0.11921108])

### TAU (MGGA)

In [135]:
result = ni.eval_xc_eff("HYB_MGGA_XC_TPSSH", rho_tau, deriv=3, spin=1)

In [136]:
[r.shape for r in result]

[(33736,), (2, 5, 33736), (2, 5, 2, 5, 33736), (2, 5, 2, 5, 2, 5, 33736)]

In [137]:
fps = np.array([
    lib.fp(result[0] * weights),
    lib.fp(result[1] * weights),
    lib.fp(result[2] * rho_tau * weights),
    lib.fp(result[3] * rho_tau * rho_tau[:, :, None, None, :] * weights),
])
fps

array([ 0.00704402,  0.09317359, -0.01473527,  1.38420532])